# 🌿 CropVision — Week 2: Custom CNN Architecture & Baseline Training
 Project 2: Agriculture & Smart Farming**

---

### 📋 Week 2 Objectives
- Build a **custom CNN architecture** from scratch (Conv2D → MaxPool → Dense)
- Train the baseline model on the `train_ds` from Week 1
- Plot **loss and accuracy curves** to monitor training
- Detect **overfitting** and fix it using Dropout + Early Stopping
- Record baseline accuracy on the validation set
- Save the best model weights for comparison in Week 3

### ⚠️ Commit Reminder
Commit **3–5 times today**. Suggested messages:
```
model: custom CNN architecture defined — 3xConv2D + MaxPool + Dense
model: baseline training complete, val_acc=0.XX
model-tuning: overfitting detected, adding Dropout(0.5)
model-tuning: EarlyStopping added, best weights restored
eval: baseline confusion matrix and classification report saved
```

---
## 0. Imports & Config

In [1]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Plot style
plt.rcParams['figure.facecolor'] = '#f9f9f9'
plt.rcParams['axes.facecolor']   = '#ffffff'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)

print('✅ Libraries imported')
print(f'   TensorFlow : {tf.__version__}')
print(f'   Keras      : {keras.__version__}')
print(f'   GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

✅ Libraries imported
   TensorFlow : 2.21.0
   Keras      : 3.14.0
   GPU available: False


---
## 1. Reload Datasets from Week 1

We rebuild the same `train_ds`, `val_ds`, `test_ds` from Week 1.  
The `class_names.json` saved last week keeps the label mapping consistent.

In [2]:
# ── Config ──
DATA_DIR = Path(r"E:\Crop-Vision\data\raw\plantvillage\grayscale")  # update if needed
IMG_SIZE   = [224, 224]
BATCH_SIZE = 32

# Load class names saved in Week 1
with open('outputs/class_names.json') as f:
    class_names = json.load(f)

NUM_CLASSES  = len(class_names)
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_label = {v: k for k, v in label_to_idx.items()}

print(f'✅ Loaded {NUM_CLASSES} classes from Week 1')


# ── Rebuild splits ──
def stratified_split(data_dir, train_r=0.70, val_r=0.15, seed=42):
    random.seed(seed)
    splits = {'train': [], 'val': [], 'test': []}
    for class_name in sorted([d.name for d in data_dir.iterdir() if d.is_dir()]):
        class_dir = data_dir / class_name
        all_imgs  = (
            list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) +
            list(class_dir.glob('*.png')) + list(class_dir.glob('*.PNG'))
        )
        random.shuffle(all_imgs)
        n       = len(all_imgs)
        n_train = int(n * train_r)
        n_val   = int(n * val_r)
        splits['train'].extend([(p, class_name) for p in all_imgs[:n_train]])
        splits['val'  ].extend([(p, class_name) for p in all_imgs[n_train:n_train + n_val]])
        splits['test' ].extend([(p, class_name) for p in all_imgs[n_train + n_val:]])
    return splits


# ── Build tf.data datasets ──
def load_and_preprocess(image_path, label_idx):
    raw = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.one_hot(label_idx, NUM_CLASSES)

def augment_tf(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.3)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(split_data, shuffle=False, augment=False):
    paths  = [str(p) for p, _ in split_data]
    labels = [label_to_idx[l] for _, l in split_data]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_tf, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


splits   = stratified_split(DATA_DIR)
train_ds = make_dataset(splits['train'], shuffle=True,  augment=True)
val_ds   = make_dataset(splits['val'],   shuffle=False, augment=False)
test_ds  = make_dataset(splits['test'],  shuffle=False, augment=False)

print(f'\n📦 Datasets ready:')
print(f'   train_ds : {len(train_ds)} batches')
print(f'   val_ds   : {len(val_ds)} batches')
print(f'   test_ds  : {len(test_ds)} batches')

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/class_names.json'

---
## 2. Build Custom CNN Architecture

### How a CNN works:
```
Input image (224×224×3)
       ↓
Conv2D — scans for edges and textures (like a magnifying glass sliding across the image)
       ↓
MaxPooling — shrinks the feature map, keeps only the strongest signals
       ↓
  ... repeat Conv2D + MaxPool layers (each learns more complex features) ...
       ↓
Flatten — converts 2D feature maps into a 1D vector
       ↓
Dense — fully connected layer that combines all features
       ↓
Dropout — randomly turns off neurons during training (prevents overfitting)
       ↓
Output Dense (softmax) — 38 probabilities, one per disease class
```

We start with a **simple 3-block CNN** as baseline. Week 3 will replace this with ResNet50.

In [ ]:
def build_custom_cnn(num_classes, input_shape=(224, 224, 3), dropout_rate=0.5):
    """
    Build a custom CNN from scratch.

    Architecture:
        Block 1: Conv2D(32)  → BatchNorm → ReLU → MaxPool
        Block 2: Conv2D(64)  → BatchNorm → ReLU → MaxPool
        Block 3: Conv2D(128) → BatchNorm → ReLU → MaxPool
        Block 4: Conv2D(256) → BatchNorm → ReLU → MaxPool
        Flatten → Dense(512) → Dropout → Output(num_classes, softmax)

    Args:
        num_classes  : number of disease classes (38 for PlantVillage)
        input_shape  : (H, W, C) — default (224, 224, 3)
        dropout_rate : fraction of neurons to randomly drop during training

    Returns:
        Compiled Keras model
    """
    model = models.Sequential([

        # ── Input ──
        layers.Input(shape=input_shape),

        # ── Block 1: Detect basic edges and colors ──
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),       # 224×224 → 112×112

        # ── Block 2: Detect simple textures ──
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),       # 112×112 → 56×56

        # ── Block 3: Detect complex patterns (spots, lesions) ──
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),       # 56×56 → 28×28

        # ── Block 4: High-level disease features ──
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),       # 28×28 → 14×14

        # ── Classifier head ──
        layers.GlobalAveragePooling2D(),   # 14×14×256 → 256 (better than Flatten)
        layers.Dense(512, activation='relu'),
        layers.Dropout(dropout_rate),      # prevent overfitting
        layers.Dense(num_classes, activation='softmax')  # output probabilities

    ], name='CropVision_CustomCNN')

    # ── Compile ──
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',   # multi-class classification loss
        metrics=['accuracy']
    )

    return model


# Build and inspect the model
model = build_custom_cnn(num_classes=NUM_CLASSES, dropout_rate=0.5)
model.summary()

In [ ]:
# Count and display parameter breakdown
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

print('📊 MODEL PARAMETER SUMMARY')
print('=' * 40)
print(f'   Total parameters     : {total_params:,}')
print(f'   Trainable parameters : {trainable_params:,}')
print(f'   Non-trainable        : {total_params - trainable_params:,}')
print()
print('🧠 ARCHITECTURE INTUITION:')
print('   Block 1 (32 filters)  → learns edges: horizontal, vertical, diagonal lines')
print('   Block 2 (64 filters)  → learns textures: rough, smooth, spotted')
print('   Block 3 (128 filters) → learns shapes: lesion outlines, discoloration patches')
print('   Block 4 (256 filters) → learns disease signatures: specific rot/blight patterns')
print('   Dense (512 units)     → combines all features into a disease profile')
print(f'   Output ({NUM_CLASSES} units)   → one probability per disease class')

---
## 3. Set Up Callbacks

**Callbacks** are functions that run automatically during training:

| Callback | What it does |
|---|---|
| `ModelCheckpoint` | Saves model weights whenever val_accuracy improves |
| `EarlyStopping` | Stops training if val_accuracy stops improving (prevents wasting time) |
| `ReduceLROnPlateau` | Reduces learning rate when training gets stuck |
| `CSVLogger` | Saves all metrics to a CSV file for analysis |

In [ ]:
# ── Callbacks ──
checkpoint_path = 'models/best_custom_cnn.keras'

cb_checkpoint = callbacks.ModelCheckpoint(
    filepath        = checkpoint_path,
    monitor         = 'val_accuracy',   # watch validation accuracy
    save_best_only  = True,             # only save when it improves
    mode            = 'max',
    verbose         = 1
)

cb_early_stop = callbacks.EarlyStopping(
    monitor              = 'val_accuracy',
    patience             = 7,           # stop if no improvement for 7 epochs
    restore_best_weights = True,        # roll back to best weights automatically
    verbose              = 1
)

cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor   = 'val_loss',
    factor    = 0.5,                    # halve the learning rate
    patience  = 3,                      # after 3 epochs of no improvement
    min_lr    = 1e-6,
    verbose   = 1
)

cb_csv_logger = callbacks.CSVLogger(
    filename  = 'outputs/training_log_cnn.csv',
    append    = False
)

all_callbacks = [cb_checkpoint, cb_early_stop, cb_reduce_lr, cb_csv_logger]

print('✅ Callbacks configured:')
print(f'   ModelCheckpoint  → saves to: {checkpoint_path}')
print(f'   EarlyStopping    → patience: 7 epochs')
print(f'   ReduceLROnPlateau→ factor: 0.5, patience: 3 epochs')
print(f'   CSVLogger        → logs to: outputs/training_log_cnn.csv')

---
## 4. Train the Baseline CNN

**What to watch during training:**
- `loss` should go **down** over epochs
- `accuracy` should go **up** over epochs
- If `val_loss` starts going UP while `loss` keeps going DOWN → **overfitting**
- If both are going down together → model is learning well

> ⏱️ Training time depends on your hardware. On CPU: ~5–10 min/epoch. On GPU: ~1–2 min/epoch.

In [ ]:
EPOCHS = 30   # EarlyStopping will stop before this if needed

print('🚀 Starting training...')
print(f'   Epochs       : {EPOCHS} (max)')
print(f'   Batch size   : {BATCH_SIZE}')
print(f'   Train batches: {len(train_ds)}')
print(f'   Val batches  : {len(val_ds)}')
print()

history = model.fit(
    train_ds,
    epochs          = EPOCHS,
    validation_data = val_ds,
    callbacks       = all_callbacks,
    verbose         = 1
)

print('\n✅ Training complete!')
print(f'   Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')
print(f'   Stopped at epoch : {len(history.history["loss"])}')

---
## 5. Plot Training Curves

**How to read these plots:**
- **Good training:** both train and val curves improve together and converge
- **Overfitting:** train accuracy keeps rising but val accuracy plateaus or drops
- **Underfitting:** both curves are low — model isn't learning enough

In [ ]:
def plot_training_curves(history, save_path='outputs/training_curves_cnn.png'):
    """
    Plot accuracy and loss curves for train and validation sets.
    Marks the best epoch with a vertical dashed line.
    """
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs   = range(1, len(acc) + 1)

    # Find best epoch (highest val_accuracy)
    best_epoch = np.argmax(val_acc) + 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Custom CNN — Training Curves', fontsize=14, fontweight='bold')

    # ── Accuracy ──
    axes[0].plot(epochs, acc,     color='#27ae60', linewidth=2, label='Train Accuracy')
    axes[0].plot(epochs, val_acc, color='#2980b9', linewidth=2, label='Val Accuracy',
                 linestyle='--')
    axes[0].axvline(best_epoch, color='#e74c3c', linestyle=':', linewidth=1.5,
                    label=f'Best epoch: {best_epoch} ({max(val_acc):.3f})')
    axes[0].set_title('Accuracy', fontsize=12)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend(fontsize=10)
    axes[0].set_ylim([0, 1])

    # ── Loss ──
    axes[1].plot(epochs, loss,     color='#e67e22', linewidth=2, label='Train Loss')
    axes[1].plot(epochs, val_loss, color='#8e44ad', linewidth=2, label='Val Loss',
                 linestyle='--')
    axes[1].axvline(best_epoch, color='#e74c3c', linestyle=':', linewidth=1.5,
                    label=f'Best epoch: {best_epoch}')
    axes[1].set_title('Loss', fontsize=12)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Saved: {save_path}')

    # Overfitting check
    final_gap = acc[-1] - val_acc[-1]
    print(f'\n🔍 OVERFITTING CHECK:')
    print(f'   Final train accuracy : {acc[-1]:.4f}')
    print(f'   Final val accuracy   : {val_acc[-1]:.4f}')
    print(f'   Gap (train - val)    : {final_gap:.4f}')
    if final_gap > 0.15:
        print('   ⚠️  Significant overfitting detected! Consider:')
        print('      → Increase Dropout rate (e.g. 0.5 → 0.6)')
        print('      → Add more data augmentation')
        print('      → Reduce model complexity')
    elif final_gap > 0.05:
        print('   ⚡ Mild overfitting — model is learning well but slightly memorizing')
    else:
        print('   ✅ Model is generalizing well — no significant overfitting!')


plot_training_curves(history)

### 📝 Observation — Training Curves

> **Write your findings here after running the cell above:**
> - What was the best validation accuracy achieved?
> - Is there a gap between train and val accuracy? (overfitting?)
> - At which epoch did EarlyStopping trigger?
> - Did the learning rate reduction help?

*Example: "Best val_accuracy = 0.74 at epoch 12. Train accuracy reached 0.91 by epoch 12 — a gap of 0.17, indicating significant overfitting. Will experiment with higher Dropout and stronger augmentation in the next run."*

---
## 6. Diagnose Overfitting & Tune

**Overfitting** = the model memorizes training images instead of learning general patterns.  
Signs: train accuracy >> val accuracy.

**Our tools to fix it:**
| Tool | How it helps |
|---|---|
| `Dropout` | Randomly disables neurons — forces the model not to rely on any single path |
| `BatchNormalization` | Stabilizes activations — already included in our CNN |
| `Data Augmentation` | More variety in training images — already applied in Week 1 |
| `EarlyStopping` | Stops before the model overfits too much — already set up |
| `L2 Regularization` | Penalizes large weights — optional extra step below |

In [ ]:
# ── Rebuild with tuned settings if overfitting was detected ──
# Increase dropout, add L2 regularization to Dense layers

from tensorflow.keras import regularizers

def build_tuned_cnn(num_classes, input_shape=(224, 224, 3),
                    dropout_rate=0.6, l2_lambda=1e-4):
    """
    Tuned version of the CNN with:
        - Higher Dropout (0.6 vs 0.5)
        - L2 regularization on Dense layers
        - Slightly lower learning rate (5e-4 vs 1e-3)
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Classifier with L2 regularization
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dropout(dropout_rate),      # higher dropout
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_lambda)),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='CropVision_TunedCNN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),  # lower lr
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Build tuned model
model_tuned = build_tuned_cnn(num_classes=NUM_CLASSES)

# Update checkpoint path
cb_checkpoint_tuned = callbacks.ModelCheckpoint(
    'models/best_tuned_cnn.keras',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)
cb_csv_tuned = callbacks.CSVLogger('outputs/training_log_tuned_cnn.csv')

print('✅ Tuned CNN built')
print(f'   Dropout rate  : 0.6 (was 0.5)')
print(f'   L2 lambda     : 1e-4')
print(f'   Learning rate : 5e-4 (was 1e-3)')
print()
print('▶️  Now training tuned model...')

history_tuned = model_tuned.fit(
    train_ds,
    epochs          = 30,
    validation_data = val_ds,
    callbacks       = [cb_checkpoint_tuned, cb_early_stop, cb_reduce_lr, cb_csv_tuned],
    verbose         = 1
)

In [ ]:
# Compare baseline vs tuned model on validation set
baseline_val_acc = max(history.history['val_accuracy'])
tuned_val_acc    = max(history_tuned.history['val_accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline CNN vs Tuned CNN — Validation Accuracy', fontsize=13)

for ax, hist, label, color in zip(
    axes,
    [history, history_tuned],
    ['Baseline CNN', 'Tuned CNN'],
    ['#27ae60', '#2980b9']
):
    e = range(1, len(hist.history['val_accuracy']) + 1)
    ax.plot(e, hist.history['accuracy'],     color=color,   linewidth=2, label='Train')
    ax.plot(e, hist.history['val_accuracy'], color=color,   linewidth=2, linestyle='--', label='Val')
    ax.set_title(f'{label}\nBest val_acc = {max(hist.history["val_accuracy"]):.4f}', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=10)
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('outputs/baseline_vs_tuned.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 COMPARISON:')
print(f'   Baseline val_accuracy : {baseline_val_acc:.4f}')
print(f'   Tuned    val_accuracy : {tuned_val_acc:.4f}')
print(f'   Improvement           : +{(tuned_val_acc - baseline_val_acc):.4f}')

# Select best model
best_model = model_tuned if tuned_val_acc >= baseline_val_acc else model
best_label = 'Tuned CNN' if tuned_val_acc >= baseline_val_acc else 'Baseline CNN'
print(f'\n   ✅ Best model: {best_label}')

---
## 7. Evaluate on Validation Set

We evaluate the best model on the **validation set** (NOT the test set — that's saved for Week 4).

**Metrics we'll compute:**
- **Accuracy** — overall % correct
- **Precision** — of all predicted as disease X, how many actually were X?
- **Recall** — of all actual disease X cases, how many did we catch?
- **F1-score** — harmonic mean of Precision and Recall
- **Confusion Matrix** — shows which diseases get confused with each other

In [ ]:
# Get predictions on validation set
print('🔍 Running predictions on validation set...')

y_true, y_pred = [], []

for img_batch, label_batch in val_ds:
    preds = best_model.predict(img_batch, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(label_batch.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Overall accuracy
val_accuracy = np.mean(y_true == y_pred)
print(f'\n✅ Validation Accuracy : {val_accuracy:.4f} ({val_accuracy*100:.2f}%)')

# Full classification report
print('\n📋 CLASSIFICATION REPORT (Validation Set):')
print('=' * 70)
report = classification_report(
    y_true, y_pred,
    target_names=[c.replace('___', ' | ').replace('_', ' ') for c in class_names]
)
print(report)

# Save report to file
with open('outputs/classification_report_cnn.txt', 'w') as f:
    f.write(f'Best model: {best_label}\n')
    f.write(f'Validation Accuracy: {val_accuracy:.4f}\n\n')
    f.write(report)
print('💾 Saved: outputs/classification_report_cnn.txt')

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Normalize to percentages
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

short_labels = [c.split('___')[-1].replace('_', ' ')[:15] for c in class_names]

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm_normalized,
    annot      = True,
    fmt        = '.2f',
    cmap       = 'YlOrRd',
    xticklabels= short_labels,
    yticklabels= short_labels,
    linewidths = 0.3,
    ax         = ax,
    annot_kws  = {'size': 6}
)
ax.set_title('Confusion Matrix — Custom CNN (Validation Set)\n'
             'Diagonal = correctly classified | Off-diagonal = misclassified',
             fontsize=13, pad=15)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.tick_params(axis='both', labelsize=8)
plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('outputs/confusion_matrix_cnn.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/confusion_matrix_cnn.png')

# Find the most confused pairs
print('\n🔍 TOP 5 MOST CONFUSED CLASS PAIRS:')
cm_no_diag = cm_normalized.copy()
np.fill_diagonal(cm_no_diag, 0)
top5 = np.unravel_index(np.argsort(cm_no_diag.ravel())[-5:][::-1], cm_no_diag.shape)
for true_idx, pred_idx in zip(top5[0], top5[1]):
    print(f'   True: {class_names[true_idx]}')
    print(f'   Pred: {class_names[pred_idx]}')
    print(f'   Rate: {cm_normalized[true_idx, pred_idx]:.2%}  ← model confuses these {cm_normalized[true_idx, pred_idx]:.0%} of the time')
    print()

### 📝 Observation — Confusion Matrix

> **Write your findings here:**
> - Which disease classes are being confused most often? Why might that be?
> - Are healthy plants being misclassified as diseased? (False positives)
> - Are diseased plants being missed entirely? (False negatives — most dangerous!)
> - What does this suggest for Week 3 (Transfer Learning)?

*Example: "Tomato Early Blight and Tomato Late Blight are the most confused pair (28% confusion rate) — visually similar brown lesion patterns. Healthy classes score highest recall. ResNet50 in Week 3 should improve feature discrimination between similar disease types."*

---
## 8. Visualize Predictions on Sample Images

In [ ]:
# Show 9 sample predictions — correct (green) and wrong (red)
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
fig.suptitle('Sample Predictions — Custom CNN (Validation Set)\n'
             '✅ Green = Correct   |   ❌ Red = Wrong', fontsize=13, y=1.01)

sample_val = list(splits['val'])[:200]
random.shuffle(sample_val)

shown = 0
for img_path, true_label in sample_val:
    if shown >= 9:
        break

    # Preprocess
    img_arr = np.array(
        Image.open(img_path).convert('RGB').resize((224, 224)),
        dtype=np.float32
    ) / 255.0
    img_input = np.expand_dims(img_arr, axis=0)  # add batch dim

    # Predict
    probs      = best_model.predict(img_input, verbose=0)[0]
    pred_idx   = np.argmax(probs)
    pred_label = class_names[pred_idx]
    confidence = probs[pred_idx]
    is_correct = pred_label == true_label

    ax = axes[shown // 3][shown % 3]
    ax.imshow(img_arr)

    true_short = true_label.split('___')[-1].replace('_', ' ')
    pred_short = pred_label.split('___')[-1].replace('_', ' ')

    title_color = '#27ae60' if is_correct else '#e74c3c'
    icon        = '✅' if is_correct else '❌'
    ax.set_title(
        f'{icon} True: {true_short}\nPred: {pred_short} ({confidence:.1%})',
        fontsize=8, color=title_color
    )
    ax.axis('off')

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(title_color)
        spine.set_linewidth(3)

    shown += 1

plt.tight_layout()
plt.savefig('outputs/sample_predictions_cnn.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/sample_predictions_cnn.png')

---
## 9. Week 2 Summary & Commit Checklist

### ✅ What we accomplished this week

| Task | Status |
|------|--------|
| Custom CNN architecture (4 blocks) | ✅ |
| Baseline training with callbacks | ✅ |
| Training curves plotted | ✅ |
| Overfitting diagnosed + fixed | ✅ |
| Tuned CNN trained | ✅ |
| Baseline vs Tuned comparison | ✅ |
| Validation confusion matrix | ✅ |
| Classification report saved | ✅ |
| Sample predictions visualized | ✅ |

### 📁 Outputs generated
```
outputs/
    training_curves_cnn.png
    baseline_vs_tuned.png
    confusion_matrix_cnn.png
    classification_report_cnn.txt
    sample_predictions_cnn.png
    training_log_cnn.csv
    training_log_tuned_cnn.csv
models/
    best_custom_cnn.keras
    best_tuned_cnn.keras
```

### 🔜 Week 3 Preview
- Load **ResNet50** pretrained on ImageNet
- Freeze base layers, add custom classification head
- Fine-tune with lower learning rate
- Target: push accuracy **above 90%**
- Compare ResNet50 vs our custom CNN

---

### 🔀 Commit these files now!
```bash
# Clear outputs first: Kernel → Restart & Clear Output

git add Week2_CropVision_CNN_Baseline.ipynb outputs/ models/
git commit -m 'model: Week 2 complete — custom CNN trained, val_acc=0.XX, overfitting fixed'
git push origin main
```

> 💡 Replace `0.XX` with your actual best val_accuracy!